# 1.1 Foundational models

This notebook introduces the LangChain building blocks you will reuse throughout the course:

1. **Chat models** — call a provider through a common interface (`init_chat_model` / provider-specific classes).
2. **Agents** — wrap a model so you can pass a list of messages and (later) tools, memory, and streaming.

You will invoke a model, inspect the response object, swap providers, create an agent, pass conversation history, and stream tokens.

## Setup

Load API keys from `.env` (for example `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`). Run this cell first.

In [ ]:
from dotenv import load_dotenv

# Reads .env in the project root and puts keys into os.environ
load_dotenv()

## Initialising and invoking a model

`init_chat_model` is the provider-agnostic way to construct a chat model. Pass a model name; LangChain infers the integration from the name (or from extra kwargs).

`invoke` sends one request and returns an **AIMessage** (not just a string). The text lives on `.content`; usage, finish reason, and provider extras live on `.response_metadata`.

In [ ]:
from langchain.chat_models import init_chat_model

# ChatOpenAI-compatible model; requires OPENAI_API_KEY
model = init_chat_model(model="gpt-5-nano")

In [ ]:
# invoke accepts a string or a list of messages; a string is treated as a HumanMessage
response = model.invoke("What's the capital of the Moon?")

# Full AIMessage: content, id, response_metadata, usage_metadata, etc.
response

In [ ]:
# Just the assistant's text (what you usually show to a user)
print(response.content)

In [ ]:
from pprint import pprint

# Provider-specific extras: token counts, model name, finish_reason, etc.
pprint(response.response_metadata)

## Customising your model

Extra keyword arguments on `init_chat_model` are forwarded to the underlying client. **Temperature** controls randomness: `0` is more deterministic; higher values (here `1.0`) are more varied. Re-run the cell and compare answers to the previous invoke.

In [ ]:
model = init_chat_model(
    model="gpt-5-nano",
    # Kwargs passed to the model:
    temperature=1.0
)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

## Model providers

The same `init_chat_model` / chat-model interface works across vendors. You can also instantiate a provider class directly (Gemini below).

Chat integrations catalogue: [LangChain chat model integrations](https://docs.langchain.com/oss/python/integrations/chat)

In [ ]:
# Anthropic via init_chat_model — needs ANTHROPIC_API_KEY
model = init_chat_model(model="claude-sonnet-4-6")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

In [ ]:
# Google Gemini via the dedicated class — needs GOOGLE_API_KEY
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

## Initialising and invoking an agent

`create_agent` builds a LangGraph agent around a model. You can pass:

- an **already-created model object**, or
- a **model name** (`model="..."` or the first positional argument).

Invoke with a dict whose `messages` key is a list of LangChain message objects. The return value is graph **state** (a dict), not a single `AIMessage`.

In [ ]:
from langchain.agents import create_agent

# Reuses the Gemini `model` from the previous cell
agent = create_agent(model=model)

In [ ]:
# Same API: pass a model id string instead of a ChatModel instance
agent = create_agent(model="claude-sonnet-4-6")

In [ ]:
# Positional model name — this is the agent used in the cells below
agent = create_agent("gpt-5-nano")

In [ ]:
from langchain.messages import HumanMessage

# Agent input is always graph state: a list of messages under "messages"
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?")]}
)

In [ ]:
from pprint import pprint

# State dict: typically includes the full message list (human + AI, and later tool messages)
pprint(response)

In [ ]:
# Last message is the agent's reply
print(response['messages'][-1].content)

### Conversation history

Chat models are **stateless** unless you send prior turns. Pass a sequence of `HumanMessage` / `AIMessage` so the model can continue the story (here, a fictional Luna City).

In [ ]:
from langchain.messages import AIMessage

# Multi-turn: the model sees the previous Q&A, then the follow-up
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]}
)

pprint(response)

## Streaming output

`agent.stream(..., stream_mode="messages")` yields `(token, metadata)` pairs as the model generates. Print chunks with `end=""` so they appear as one live sentence instead of one line per token.

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token
    
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token